In [16]:
# # ProctorIQ Assessment Journey RAG Challenge - Starter Notebook

# ProctorIQ runs proctored assessments for certification programs. Build a RAG-powered chatbot
# that resolves student issues (installation, network, device, camera/integrity, post-exam
# status), cites the document/section it pulled the answer from, and never assists in evading
# proctoring integrity checks.

# ## Submission Format

# `submission.csv`:
# # 
# | Column | Description |
# |---|---|
# | `question_id` | from `test.csv`, unchanged |
# | `answer_text` | your pipeline's final answer |
# | `cited_docs` | source document filename(s), pipe-separated if more than one |
# | `cited_sections` | section name(s), pipe-separated, aligned by position with `cited_docs` |

# `# TODO` cells are yours to complete. Keep the rest of the structure intact.

## Step 0: Setup

In [17]:
!pip install -q groq python-dotenv chromadb sentence-transformers \
    langchain-core langchain-community langchain-text-splitters \
    faiss-cpu rank-bm25

In [18]:
import os
import glob
import json
import time
from pathlib import Path

import pandas as pd
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Locate the knowledge base and test.csv without guessing paths ──────────
def find_file_dir(filename_pattern):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if Path(f).match(filename_pattern):
                return root
    return None

KB_DIR = find_file_dir("*.md")
TEST_CSV_DIR = find_file_dir("test.csv")

if KB_DIR is None:
    raise FileNotFoundError("Knowledge base .md files not found under /kaggle/input — check your Add Input.")
if TEST_CSV_DIR is None:
    raise FileNotFoundError("test.csv not found under /kaggle/input — check your Add Input.")

print(f"Knowledge base found at: {KB_DIR}")
print(f"test.csv found at: {TEST_CSV_DIR}")

TEST_CSV_PATH = os.path.join(TEST_CSV_DIR, "test.csv")
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Groq API key — set this via Kaggle Secrets (Add-ons > Secrets), never hardcode it
from kaggle_secrets import UserSecretsClient
GROQ_API_KEY = UserSecretsClient().get_secret("gorq_api_key")
DEFAULT_GROQ_MODEL_ID = "qwen/qwen3.8-27b"


Knowledge base found at: /kaggle/input/datasets/sameerkampa/md-files
test.csv found at: /kaggle/input/datasets/sameerkampa/test-csv


In [19]:
# ── Load all knowledge base documents ────────────────────────────────────
def load_documents(kb_dir):
    docs = {}
    for path in sorted(glob.glob(os.path.join(kb_dir, "*.md"))):
        doc_id = Path(path).stem  # e.g. "04_network_connectivity_issue_playbook"
        with open(path, "r", encoding="utf-8") as f:
            docs[doc_id] = f.read()
    return docs

documents = load_documents(KB_DIR)
print(f"Loaded {len(documents)} documents:")
for doc_id in documents:
    print(f"  - {doc_id} ({len(documents[doc_id].split())} words)")


Loaded 10 documents:
  - 01_windows_installation_login_guide (552 words)
  - 02_mac_installation_login_guide (566 words)
  - 03_mock_test_device_readiness_guide (451 words)
  - 04_network_connectivity_issue_playbook (371 words)
  - 05_device_system_issue_playbook (324 words)
  - 06_camera_microphone_face_verification_guide (360 words)
  - 07_permitted_prohibited_actions_policy (435 words)
  - 08_reattempt_request_policy (312 words)
  - 09_status_verification_outcome_guide (357 words)
  - 10_general_support_escalation_guide (353 words)


---
## TODO 1: Chunking & Embedding Configuration

Test at least 2 chunk sizes and at least 2 embedding models before settling on a final
configuration. Show your comparison, then write a brief justification for the config you
picked in the markdown cell below.

In [20]:
# TODO: Test at least 2 chunk sizes and at least 2 embedding models, compare results.

# --- your comparison code here ---

# TODO: Test at least 2 chunk sizes and at least 2 embedding models
import numpy as np
import pandas as pd
import time

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Configurations to test
chunk_sizes = [300, 500]
chunk_overlap = 50

embedding_models = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
]

# 2. Test queries
test_queries = [
    "How to fix camera and microphone permission issues?",
    "What actions are prohibited during a mock test?",
    "What should I do if my internet connection fails?",
    "What is the reattempt policy?"
]

results = []

# 3. Compare configurations
for size in chunk_sizes:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=chunk_overlap
    )
    chunks = []
    for doc_name, content in documents.items():
        for chunk in splitter.split_text(content):
            chunks.append({
                "text": chunk,
                "source": doc_name
            })
    texts = [chunk["text"] for chunk in chunks]
    for model_name in embedding_models:
        print(f"\nTesting chunk size={size}, model={model_name}")

        start = time.time()

        model = SentenceTransformer(model_name)

        embeddings = model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        embedding_time = time.time() - start

        query_scores = []

        for query in test_queries:

            query_embedding = model.encode(
                [query],
                normalize_embeddings=True
            )[0]

            similarities = np.dot(embeddings, query_embedding)

            top_indices = np.argsort(similarities)[::-1][:3]

            query_scores.append(
                np.mean(similarities[top_indices])
            )

        avg_score = float(np.mean(query_scores))

        results.append({
            "Chunk Size": size,
            "Chunk Overlap": chunk_overlap,
            "Embedding Model": model_name,
            "Total Chunks": len(chunks),
            "Embedding Dimensions": embeddings.shape[1],
            "Embedding Time": round(embedding_time, 2),
            "Average Top-3 Similarity": round(avg_score, 4)
        })

        del model

# 4. Display comparison
comparison_df = pd.DataFrame(results)

display(
    comparison_df.sort_values(
        "Average Top-3 Similarity",
        ascending=False
    )
)

# 5. Select the best configuration based on comparison score
best_result = comparison_df.loc[
    comparison_df["Average Top-3 Similarity"].idxmax()
]

# --- set your final chosen config here, used by the rest of the notebook ---
CHUNK_SIZE = int(best_result["Chunk Size"])
CHUNK_OVERLAP = int(best_result["Chunk Overlap"])
EMBEDDING_MODEL_NAME = best_result["Embedding Model"]

print("\nSelected Configuration:")
print("CHUNK_SIZE =", CHUNK_SIZE)
print("CHUNK_OVERLAP =", CHUNK_OVERLAP)
print("EMBEDDING_MODEL_NAME =", EMBEDDING_MODEL_NAME)

assert CHUNK_SIZE is not None, "Set CHUNK_SIZE above"
assert EMBEDDING_MODEL_NAME is not None, "Set EMBEDDING_MODEL_NAME above"


Testing chunk size=300, model=sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Testing chunk size=300, model=sentence-transformers/multi-qa-MiniLM-L6-cos-v1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Testing chunk size=500, model=sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Testing chunk size=500, model=sentence-transformers/multi-qa-MiniLM-L6-cos-v1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Chunk Size,Chunk Overlap,Embedding Model,Total Chunks,Embedding Dimensions,Embedding Time,Average Top-3 Similarity
0,300,50,sentence-transformers/all-MiniLM-L6-v2,155,384,1.54,0.5689
1,300,50,sentence-transformers/multi-qa-MiniLM-L6-cos-v1,155,384,1.45,0.5669
2,500,50,sentence-transformers/all-MiniLM-L6-v2,73,384,1.48,0.5057
3,500,50,sentence-transformers/multi-qa-MiniLM-L6-cos-v1,73,384,1.51,0.4922



Selected Configuration:
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50
EMBEDDING_MODEL_NAME = sentence-transformers/all-MiniLM-L6-v2


---
## Step 2: Build the Vector Store (Provided)

In [21]:
import faiss

def chunk_documents(documents, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunk_records = []
    for doc_id, text in documents.items():
        for part in text.split("\n## "):
            if not part.strip():
                continue
            lines = part.strip().split("\n", 1)
            section_title = lines[0].strip().lstrip("#").strip()
            section_text = lines[1] if len(lines) > 1 else ""
            for chunk in splitter.split_text(section_text):
                chunk_records.append({"doc_id": doc_id, "section": section_title, "text": chunk})
    return chunk_records

chunk_records = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
chunk_embeddings = embedder.encode([c["text"] for c in chunk_records], normalize_embeddings=True)

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(np.array(chunk_embeddings, dtype="float32"))

def retrieve(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    _, indices = index.search(np.array(q_emb, dtype="float32"), k)
    return [chunk_records[i] for i in indices[0]]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:

import faiss

def chunk_documents(documents, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunk_records = []

    for doc_id, text in documents.items():
        for part in text.split("\n## "):
            if not part.strip():
                continue

            lines = part.strip().split("\n", 1)
            section_title = lines[0].strip().lstrip("#").strip()
            section_text = lines[1] if len(lines) > 1 else ""

            if not section_text.strip():
                section_text = part.strip()

            for chunk in splitter.split_text(section_text):
                chunk_records.append({
                    "doc_id": doc_id,
                    "section": section_title,
                    "text": chunk
                })

    return chunk_records


chunk_records = chunk_documents(
    documents, CHUNK_SIZE, CHUNK_OVERLAP
)

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_embeddings = embedder.encode(
    [c["text"] for c in chunk_records],
    normalize_embeddings=True,
    show_progress_bar=True
)

chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)


def retrieve(query, k=5):
    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True
    )

    q_emb = np.asarray(q_emb, dtype="float32")

    k = min(k, index.ntotal)

    scores, indices = index.search(q_emb, k)

    results = []

    for idx, score in zip(indices[0], scores[0]):
        if idx == -1:
            continue

        record = chunk_records[idx].copy()
        record["score"] = float(score)
        results.append(record)

    return results


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [23]:
print("Total chunks:", len(chunk_records))
print("Total FAISS vectors:", index.ntotal)

results = retrieve(
    "How can I fix camera and microphone permission issues?",
    k=5
)

for result in results:
    print("\nDocument:", result["doc_id"])
    print("Section:", result["section"])
    print("Score:", result["score"])
    print("Text:", result["text"][:200])

Total chunks: 125
Total FAISS vectors: 125

Document: 06_camera_microphone_face_verification_guide
Section: Section 1: Responding to a Warning
Score: 0.666002094745636
Text: - Verify that camera and microphone permissions are still enabled for PSB (permissions can occasionally be revoked by an OS update mid-session on some devices).
- Follow any on-screen instructions dis

Document: 02_mac_installation_login_guide
Section: Section 1: Installing PSB on macOS
Score: 0.5022938251495361
Text: 4. Grant PSB the required permissions when prompted: Camera, Microphone, Screen Recording, and Accessibility. All four must be enabled — PSB will not proceed to login if any permission is denied.
5. R

Document: 03_mock_test_device_readiness_guide
Section: Section 3: Common Mock-Test Failures and Fixes
Score: 0.49678048491477966
Text: ### Webcam Not Detected
- Confirm no other application (video calling software, other browser tabs) is currently using your camera — most webcams can only be accessed by

---
## Step 3: LLM Class (Provided)

Note: Groq's free tier has real rate limits (RPM/TPM/RPD) — you'll need to handle that
yourself if your pipeline runs into it.

In [24]:
class LLM:
    def __init__(self, llm_params=None, model_id=DEFAULT_GROQ_MODEL_ID):
        self.client = Groq(api_key=GROQ_API_KEY)
        self.model_id = model_id
        self.llm_params = llm_params or {"temperature": 0.2, "max_tokens": 500}

    def generate_response(self, prompt):
        completion = self.client.chat.completions.create(
            model=self.model_id,
            messages=[{"role": "user", "content": prompt}],
            **self.llm_params,
        )
        return completion.choices[0].message.content


llm_obj = LLM()
print("LLM class ready.")

LLM class ready.


---
## TODO 2: Reranker

In [25]:

class Reranker:
    def __init__(self, model_name="BAAI/bge-reranker-base"):
        # Load the CrossEncoder model
        self.model = CrossEncoder(model_name)

    def rerank(self, chunks, question, top_k=5):
        if not chunks:
            return []

        # Create (question, chunk_text) pairs
        pairs = [
            (question, chunk["text"])
            for chunk in chunks
        ]

        # Calculate relevance scores
        scores = self.model.predict(pairs)

        # Attach scores to chunks
        scored_chunks = []

        for chunk, score in zip(chunks, scores):
            result = chunk.copy()
            result["rerank_score"] = float(score)
            scored_chunks.append(result)

        # Sort by relevance score (highest first)
        scored_chunks.sort(
            key=lambda x: x["rerank_score"],
            reverse=True
        )

        # Return top-k chunks
        return scored_chunks[:top_k]


# Create reranker object
reranker = Reranker()

print("Reranker ready!")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reranker ready!


---
## TODO 3: HyDE (Hypothetical Document Embeddings)

In [26]:

class HyDERetriever:
    def __init__(self, llm_obj):
        self.llm_obj = llm_obj

    def _generate_hypothetical_answer(self, query):
        # Generate a hypothetical answer using the LLM
        prompt = f"""
        Write a short and informative hypothetical answer
        to the following question.

        Question: {query}

        Provide a possible answer in 3-5 sentences.
        """

        answer = self.llm_obj.generate_response(prompt)

        return answer

    def retrieve(self, query, k=5):
        # 1. Generate hypothetical answer
        hypothetical_answer = self._generate_hypothetical_answer(query)

        # 2. Convert hypothetical answer into an embedding
        query_embedding = embedder.encode(
            [hypothetical_answer],
            normalize_embeddings=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype="float32"
        )

        # 3. Search FAISS index
        k = min(k, index.ntotal)

        scores, indices = index.search(query_embedding, k)

        # 4. Return retrieved chunks
        results = []

        for idx, score in zip(indices[0], scores[0]):
            if idx == -1:
                continue

            result = chunk_records[idx].copy()
            result["hyde_score"] = float(score)
            results.append(result)

        return results


# Create HyDE retriever
hyde_retriever = HyDERetriever(llm_obj)

print("HyDE Retriever ready!")



HyDE Retriever ready!


---
## TODO 4: RAG Fusion

In [27]:
class RAGFusion:
    def __init__(self, llm_obj, n_queries=3):
        self.llm_obj = llm_obj
        self.n_queries = n_queries

    def _generate_queries(self, question):
        prompt = f"""
        Generate {self.n_queries} different search queries by rephrasing
        the following question.

        Keep the original meaning and focus on different relevant keywords.
        Return only the queries, one per line. Do not include numbering.

        Question: {question}
        """

        response = self.llm_obj.generate_response(prompt)

        queries = [
            line.strip().lstrip("0123456789.-) ")
            for line in response.split("\n")
            if line.strip()
        ]

        return queries[:self.n_queries]

    def _reciprocal_rank_fusion(self, results_per_query, k=60):
        fused_scores = {}
        chunk_data = {}

        for results in results_per_query:
            for rank, chunk in enumerate(results, start=1):
                chunk_id = (
                    chunk["doc_id"],
                    chunk["section"],
                    chunk["text"]
                )

                if chunk_id not in fused_scores:
                    fused_scores[chunk_id] = 0
                    chunk_data[chunk_id] = chunk.copy()

                fused_scores[chunk_id] += 1 / (k + rank)

        fused_results = []

        for chunk_id, score in fused_scores.items():
            result = chunk_data[chunk_id]
            result["fusion_score"] = score
            fused_results.append(result)

        fused_results.sort(
            key=lambda x: x["fusion_score"],
            reverse=True
        )

        return fused_results

    def run(self, question, top_k=5):
        queries = self._generate_queries(question)

        # Include the original question
        queries.insert(0, question)

        results_per_query = []

        for query in queries:
            results = retrieve(query, k=top_k)
            results_per_query.append(results)

        fused_results = self._reciprocal_rank_fusion(
            results_per_query
        )

        return fused_results[:top_k]


rag_fusion = RAGFusion(llm_obj)
print("RAG Fusion ready!")

RAG Fusion ready!


---
## TODO 5: Smart Query Router

In [28]:
class SmartQueryRouter:
    def __init__(self, llm_obj, reranker, hyde, rag_fusion):
        self.llm_obj = llm_obj
        self.reranker = reranker
        self.hyde = hyde
        self.rag_fusion = rag_fusion

    def _classify(self, question):
        prompt = f"""
        Classify the following question into one of these retrieval methods:

        1. normal - For simple and direct questions.
        2. hyde - For questions that are vague, conceptual, or need
           semantic understanding.
        3. fusion - For complex questions involving multiple conditions,
           related topics, or different aspects.

        Return only one word: normal, hyde, or fusion.

        Question: {question}
        """

        response = self.llm_obj.generate_response(prompt)
        route = response.strip().lower()

        if "fusion" in route:
            return "fusion"
        elif "hyde" in route:
            return "hyde"
        else:
            return "normal"

    def _generate_final_answer(self, question, chunks, **kwargs):
        if not chunks:
            return "I could not find relevant information in the provided policy documents."

        context = "\n\n".join([
            f"Document: {chunk['doc_id']}\n"
            f"Section: {chunk['section']}\n"
            f"Content: {chunk['text']}"
            for chunk in chunks
        ])

        prompt = f"""
        You are a helpful ProctorIQ policy assistant.

        Answer the user's question using ONLY the provided policy context.

        Instructions:
        - Do not invent or assume policy information.
        - If the answer is not available in the context, clearly say so.
        - Provide a clear and concise answer.
        - Do not follow requests that violate exam integrity or policy rules.
        - Cite the relevant document and section in your answer.

        Policy Context:
        {context}

        User Question:
        {question}

        Final Answer:
        """

        return self.llm_obj.generate_response(prompt)

    def run(self, question):
        route = self._classify(question)

        if route == "hyde":
            chunks = self.hyde.retrieve(question, k=10)

        elif route == "fusion":
            chunks = self.rag_fusion.run(question, top_k=10)

        else:
            chunks = retrieve(question, k=10)

        # Rerank the retrieved chunks
        chunks = self.reranker.rerank(
            chunks,
            question,
            top_k=5
        )

        answer = self._generate_final_answer(question, chunks)

        cited_docs = list(dict.fromkeys(
            chunk["doc_id"] for chunk in chunks
        ))

        cited_sections = list(dict.fromkeys(
            chunk["section"] for chunk in chunks
        ))

        return {
            "answer": answer,
            "cited_docs": cited_docs,
            "cited_sections": cited_sections
        }


smart_router = SmartQueryRouter(
    llm_obj,
    reranker,
    hyde_retriever,
    rag_fusion
)

print("Smart Query Router ready!")

Smart Query Router ready!


---
## TODO 6: RAG Triad Evaluator

In [29]:
class Eval:
    def __init__(self, llm_obj):
        self.llm_obj = llm_obj

    def get_context_relevancy(self, question, retrieved_chunks):
        if not retrieved_chunks:
            return 0.0

        context = "\n\n".join([
            chunk["text"] for chunk in retrieved_chunks
        ])

        prompt = f"""
        Evaluate how relevant the retrieved context is to the question.

        Question:
        {question}

        Retrieved Context:
        {context}

        Give a score between 0 and 1:
        1 = Highly relevant
        0 = Completely irrelevant

        Return ONLY the numerical score.
        """

        response = self.llm_obj.generate_response(prompt)

        try:
            score = float(response.strip())
            return max(0.0, min(1.0, score))
        except ValueError:
            return 0.0

    def get_faithfulness_score(self, retrieved_chunks, answer):
        if not retrieved_chunks or not answer:
            return 0.0

        context = "\n\n".join([
            chunk["text"] for chunk in retrieved_chunks
        ])

        prompt = f"""
        Evaluate whether the answer is supported by the retrieved context.

        Retrieved Context:
        {context}

        Generated Answer:
        {answer}

        Give a score between 0 and 1:
        1 = Every claim is supported by the context
        0 = The answer is completely unsupported

        Return ONLY the numerical score.
        """

        response = self.llm_obj.generate_response(prompt)

        try:
            score = float(response.strip())
            return max(0.0, min(1.0, score))
        except ValueError:
            return 0.0

    def get_answer_relevancy(self, question, answer):
        if not answer:
            return 0.0

        prompt = f"""
        Evaluate how well the answer addresses the question.

        Question:
        {question}

        Answer:
        {answer}

        Give a score between 0 and 1:
        1 = Directly and completely answers the question
        0 = Does not answer the question

        Return ONLY the numerical score.
        """

        response = self.llm_obj.generate_response(prompt)

        try:
            score = float(response.strip())
            return max(0.0, min(1.0, score))
        except ValueError:
            return 0.0


evaluator = Eval(llm_obj)
print("RAG Triad Evaluator ready!")

RAG Triad Evaluator ready!


---
## Step 4: Generate Submission (Provided)

In [30]:
test_df = pd.read_csv(TEST_CSV_PATH)
print(f"Loaded {len(test_df)} test questions")

reranker_obj = Reranker()
hyde_obj = HyDERetriever(llm_obj)
rag_fusion_obj = RAGFusion(llm_obj)
router = SmartQueryRouter(llm_obj, reranker_obj, hyde_obj, rag_fusion_obj)

results = []
for _, row in test_df.iterrows():
    qid, question = row["question_id"], row["question"]
    try:
        result = router.run(question)
        answer = result["answer"]
        cited_docs = "|".join(result["cited_docs"])
        cited_sections = "|".join(result["cited_sections"])
    except Exception as e:
        print(f"  [{qid}] ERROR: {e}")
        answer, cited_docs, cited_sections = "", "", ""
    results.append({
        "question_id": qid,
        "answer_text": answer,
        "cited_docs": cited_docs,
        "cited_sections": cited_sections,
    })
    print(f"  [{qid}] {question[:70]}{'...' if len(question) > 70 else ''}")
    print(f"       -> answer: {answer[:200]}{'...' if len(answer) > 200 else ''}")
    print(f"       -> cited: {cited_docs} | {cited_sections}")
    print()

submission_df = pd.DataFrame(results)
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"\nsubmission.csv written to {SUBMISSION_PATH}")


Loaded 50 test questions


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  [Q01] I'm on Windows and PSB installation failed with 'Element not found', w...
       -> answer: Based on the **01_windows_installation_login_guide**, Section 2: Common Installation Errors, follow these steps to resolve the "Element not found" error:

1.  Uninstall PSB completely via **Control Pa...
       -> cited: 01_windows_installation_login_guide|02_mac_installation_login_guide | Section 2: Common Installation Errors|Section 1: Installing PSB on Windows

  [Q02] PSB shows 'Session Start Error' on my Windows laptop, how do I resolve...
       -> answer: To resolve the "Session Start Error" on a Windows laptop, follow these steps based on the **01_windows_installation_login_guide**, **Section 2: Common Installation Errors**:

1.  **Check for Blocking ...
       -> cited: 01_windows_installation_login_guide|02_mac_installation_login_guide | Section 2: Common Installation Errors

  [Q03] I got an 'Unspecified Error' after installing PSB on Windows, what's t...
       -> answer: To 

In [31]:
import os

print(os.path.exists("/kaggle/working/submission.csv"))
print(os.path.getsize("/kaggle/working/submission.csv"))

True
47676


In [32]:
from IPython.display import FileLink, display

display(FileLink('/kaggle/working/submission.csv'))

/kaggle/working/submission.csv

In [33]:
import pandas as pd

# Load the latest submission
df = pd.read_csv(SUBMISSION_PATH, keep_default_na=False)

# Convert all required columns to strings
required_columns = [
    "question_id",
    "answer_text",
    "cited_docs",
    "cited_sections"
]

for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

# Check every question automatically
for idx, row in df.iterrows():

    qid = row["question_id"]

    # Identify missing fields
    missing_answer = not row["answer_text"].strip()
    missing_docs = not row["cited_docs"].strip()
    missing_sections = not row["cited_sections"].strip()

    # Skip complete rows
    if not (missing_answer or missing_docs or missing_sections):
        continue

    print(f"\nFixing {qid}...")

    # Get the original question
    question_row = test_df[test_df["question_id"].astype(str) == qid]

    if question_row.empty:
        print(f"Question {qid} not found in test data!")
        continue

    question = question_row.iloc[0]["question"]

    try:
        result = router.run(question)

        # Fill only missing fields
        answer = str(result.get("answer") or "").strip()
        docs = "|".join(result.get("cited_docs") or [])
        sections = "|".join(result.get("cited_sections") or [])

        if missing_answer and answer:
            df.at[idx, "answer_text"] = answer

        if missing_docs and docs.strip():
            df.at[idx, "cited_docs"] = docs

        if missing_sections and sections.strip():
            df.at[idx, "cited_sections"] = sections

        print(f"{qid} processed successfully!")

    except Exception as e:
        print(f"{qid} failed: {e}")

# Final validation
for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

print("\n========== FINAL VALIDATION ==========")
print("Total questions:", len(df))

print("\nEmpty values:")
print(df[required_columns].eq("").sum())

# Show any remaining incomplete rows
incomplete = df[
    df[required_columns].eq("").any(axis=1)
]

if incomplete.empty:
    print("\nAll questions have answers and citations!")

    df.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")
    print("Updated submission.csv saved successfully!")
else:
    print("\nSome questions still need attention:")
    display(incomplete)
    print("Do not submit until these rows are fixed.")


========== FINAL VALIDATION ==========
Total questions: 50

Empty values:
question_id       0
answer_text       0
cited_docs        0
cited_sections    0
dtype: int64

All questions have answers and citations!
Updated submission.csv saved successfully!


In [34]:
import os
import pandas as pd

print("Current submission:", SUBMISSION_PATH)
print("File exists:", os.path.exists(SUBMISSION_PATH))

if os.path.exists(SUBMISSION_PATH):
    old_df = pd.read_csv(SUBMISSION_PATH, keep_default_na=False)

    print("Rows:", len(old_df))
    print("\nNon-empty answers:",
          old_df["answer_text"].str.strip().ne("").sum())

    display(old_df.head(10))

Current submission: /kaggle/working/submission.csv
File exists: True
Rows: 50

Non-empty answers: 50


,question_id,answer_text,cited_docs,cited_sections
0,Q01,Based on the **01_windows_installation_login_g...,01_windows_installation_login_guide|02_mac_ins...,Section 2: Common Installation Errors|Section ...
1,Q02,"To resolve the ""Session Start Error"" on a Wind...",01_windows_installation_login_guide|02_mac_ins...,Section 2: Common Installation Errors
2,Q03,"To resolve the ""Unspecified Error"" on Windows,...",01_windows_installation_login_guide,Section 2: Common Installation Errors|Section ...
3,Q04,To resolve login issues caused by being redire...,01_windows_installation_login_guide|02_mac_ins...,Section 3: Login Issues|Section 1: Installing ...
4,Q05,"To fix the ""Element not found"" error on your M...",01_windows_installation_login_guide|02_mac_ins...,Section 2: Common Installation Errors|Section ...
5,Q06,"No, you do not need a different installer. Acc...",02_mac_installation_login_guide|01_windows_ins...,Section 4: Apple Silicon Note|Overview|Section...
6,Q07,You should confirm that your registered contac...,02_mac_installation_login_guide|01_windows_ins...,Section 3: Login Issues|Section 5: Before You ...
7,Q08,"No, you do not need to contact support. A forg...",02_mac_installation_login_guide|01_windows_ins...,Section 3: Login Issues|Section 2: Common Inst...
8,Q09,If your webcam is not detected during your moc...,03_mock_test_device_readiness_guide|01_windows...,Section 3: Common Mock-Test Failures and Fixes...
9,Q10,To fix the issue of your face not being clearl...,03_mock_test_device_readiness_guide|06_camera_...,Section 3: Common Mock-Test Failures and Fixes...


In [35]:
# Check whether the RAG pipeline is working

print("Testing question:", test_df.iloc[1]["question"])

try:
    test_result = router.run(test_df.iloc[1]["question"])

    print("\nANSWER:")
    print(test_result.get("answer", "NO ANSWER"))

    print("\nCITED DOCUMENTS:")
    print(test_result.get("cited_docs", []))

    print("\nCITED SECTIONS:")
    print(test_result.get("cited_sections", []))

except Exception as e:
    print("\nROUTER ERROR:")
    print(type(e).__name__, str(e))

Testing question: PSB shows 'Session Start Error' on my Windows laptop, how do I resolve it?

ANSWER:
To resolve the "Session Start Error" on a Windows laptop, follow these steps based on the **01_windows_installation_login_guide** (Section 2: Common Installation Errors):

1.  **Check for Blocking Software:** The error is usually caused by a firewall or antivirus blocking the connection to the ProctorIQ assessment server. Ensure these are not blocking PSB.
2.  **Restart and Reinstall:**
    *   Restart your laptop.
    *   Re-download the installer fresh (do not reuse the old installer file).
    *   Install again as Administrator, then restart your laptop once more before opening PSB.
3.  **Retry Login:**
    *   Reopen PSB and attempt to log in again.
    *   If the error persists after two attempts, restart your laptop before trying a third time.

**Source:** Document: 01_windows_installation_login_guide, Section 2: Common Installation Errors.

CITED DOCUMENTS:
['01_windows_installa

In [36]:
import pandas as pd
import time
import os

SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Start with an existing valid submission if one exists;
# otherwise create a fresh dataframe.
if os.path.exists(SUBMISSION_PATH):
    results_df = pd.read_csv(
        SUBMISSION_PATH,
        keep_default_na=False
    )
else:
    results_df = pd.DataFrame({
        "question_id": test_df["question_id"],
        "answer_text": "",
        "cited_docs": "",
        "cited_sections": ""
    })

# Ensure all test questions are represented
results_df = test_df[["question_id"]].merge(
    results_df,
    on="question_id",
    how="left"
).fillna("")

for _, row in test_df.iterrows():

    qid = row["question_id"]
    question = row["question"]

    existing = results_df[
        results_df["question_id"] == qid
    ].index[0]

    # Skip questions already completed
    if (
        str(results_df.at[existing, "answer_text"]).strip()
        and str(results_df.at[existing, "cited_docs"]).strip()
        and str(results_df.at[existing, "cited_sections"]).strip()
    ):
        print(f"{qid}: Already completed")
        continue

    print(f"\nProcessing {qid}...")

    for attempt in range(3):
        try:
            result = router.run(question)

            answer = str(result.get("answer", "")).strip()
            docs = result.get("cited_docs", [])
            sections = result.get("cited_sections", [])

            docs = "|".join(map(str, docs)) if isinstance(docs, list) else str(docs)
            sections = "|".join(map(str, sections)) if isinstance(sections, list) else str(sections)

            if not answer or not docs or not sections:
                raise ValueError("Answer or citations are missing")

            results_df.at[existing, "answer_text"] = answer
            results_df.at[existing, "cited_docs"] = docs
            results_df.at[existing, "cited_sections"] = sections

            # Save after every successful question
            results_df.to_csv(SUBMISSION_PATH, index=False)

            print(f"{qid}: Completed")
            break

        except Exception as e:
            print(f"{qid}: Attempt {attempt + 1} failed: {e}")

            if attempt < 2:
                time.sleep(10)

    else:
        print(f"{qid}: Skipped after 3 failed attempts")

print("\n========== FINAL VALIDATION ==========")

final_df = pd.read_csv(
    SUBMISSION_PATH,
    keep_default_na=False
)

print("Total questions:", len(final_df))
print("Completed answers:", final_df["answer_text"].str.strip().ne("").sum())
print("Missing answers:", final_df["answer_text"].str.strip().eq("").sum())
print("Missing documents:", final_df["cited_docs"].str.strip().eq("").sum())
print("Missing sections:", final_df["cited_sections"].str.strip().eq("").sum())

display(final_df.head())

print("\nSaved at:", SUBMISSION_PATH)

Q01: Already completed
Q02: Already completed
Q03: Already completed
Q04: Already completed
Q05: Already completed
Q06: Already completed
Q07: Already completed
Q08: Already completed
Q09: Already completed
Q10: Already completed
Q11: Already completed
Q12: Already completed
Q13: Already completed
Q14: Already completed
Q15: Already completed
Q16: Already completed
Q17: Already completed
Q18: Already completed
Q19: Already completed
Q20: Already completed
Q21: Already completed
Q22: Already completed
Q23: Already completed
Q24: Already completed
Q25: Already completed
Q26: Already completed
Q27: Already completed
Q28: Already completed
Q29: Already completed
Q30: Already completed
Q31: Already completed
Q32: Already completed
Q33: Already completed
Q34: Already completed
Q35: Already completed
Q36: Already completed
Q37: Already completed
Q38: Already completed
Q39: Already completed
Q40: Already completed
Q41: Already completed
Q42: Already completed
Q43: Already completed
Q44: Alread

,question_id,answer_text,cited_docs,cited_sections
0,Q01,Based on the **01_windows_installation_login_g...,01_windows_installation_login_guide|02_mac_ins...,Section 2: Common Installation Errors|Section ...
1,Q02,"To resolve the ""Session Start Error"" on a Wind...",01_windows_installation_login_guide|02_mac_ins...,Section 2: Common Installation Errors
2,Q03,"To resolve the ""Unspecified Error"" on Windows,...",01_windows_installation_login_guide,Section 2: Common Installation Errors|Section ...
3,Q04,To resolve login issues caused by being redire...,01_windows_installation_login_guide|02_mac_ins...,Section 3: Login Issues|Section 1: Installing ...
4,Q05,"To fix the ""Element not found"" error on your M...",01_windows_installation_login_guide|02_mac_ins...,Section 2: Common Installation Errors|Section ...



Saved at: /kaggle/working/submission.csv


In [37]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/submission.csv"))

/kaggle/working/submission.csv

---
## Before You Submit

- [ ] Ran the full notebook top to bottom with **Save & Run All (Commit)**
- [ ] `submission.csv` has one row per `test.csv` question, in the required format
- [ ] Submitted from this notebook's committed Output, not an uploaded CSV